Import libraries and set paths

In [2]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_validate, cross_val_predict

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

project_dir = Path(
    r"C:\Users\U S E R\Desktop\ML Assigment\project"
)

models_dir = project_dir / "models"
results_dir = (
    project_dir / "data" / "processed" / "model_comparisons"
)

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

Load processed training data

In [5]:
train_df = pd.read_csv(
    project_dir / "data" / "processed" /
    "math_train_processed.csv"
)

X_train = train_df.drop(columns=["G3"])
y_train = train_df["G3"]

assert X_train.notna().all().all()
assert y_train.notna().all()

print("Training records:", len(X_train))
print("Features:", X_train.shape[1])
train_df.head()

Training records: 316
Features: 58


,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,...,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes,G3
0,-0.5,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,12
1,-0.5,-0.5,0.0,1.0,-1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,14
2,-0.5,-1.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0
3,-0.5,0.5,0.0,0.0,1.0,0.0,-1.0,1.0,0.5,1.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,9
4,-0.5,-0.5,-2.0,1.0,-1.0,0.0,-1.0,0.0,-0.5,0.0,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,8


Define experiments and cross-validation

In [6]:
rf_experiments = {
    "RF-1": {
        "n_estimators": 100,
        "max_depth": None,
        "min_samples_leaf": 1
    },
    "RF-2": {
        "n_estimators": 200,
        "max_depth": 5,
        "min_samples_leaf": 1
    },
    "RF-3": {
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_leaf": 2
    },
    "RF-4": {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 2
    },
    "RF-5": {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 4
    }
}

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

display(pd.DataFrame.from_dict(rf_experiments, orient="index"))

,n_estimators,max_depth,min_samples_leaf
RF-1,100,NaN,1
RF-2,200,5.0,1
RF-3,200,10.0,2
RF-4,300,15.0,2
RF-5,300,NaN,4


Train and evaluate all Random Forest configurations

In [8]:
rf_results = []
rf_fold_results = []

actual = y_train.to_numpy()
actual_binary = (actual >= 10).astype(int)

for name, params in rf_experiments.items():

    print(f"Training {name}...")

    model = RandomForestRegressor(
        **params,
        random_state=42,
        n_jobs=-1
    )

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1
    )

    predictions = cross_val_predict(
        model,
        X_train,
        y_train,
        cv=cv,
        n_jobs=1
    )

    rounded_predictions = np.clip(
        np.rint(predictions), 0, 20
    ).astype(int)

    predicted_binary = (predictions >= 10).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    train_rmse = np.sqrt(-scores["train_MSE"]).mean()
    val_rmse = np.sqrt(-scores["test_MSE"]).mean()

    rf_results.append({
        "Experiment": name,
        "Trees": params["n_estimators"],
        "Max Depth": str(params["max_depth"]),
        "Min Leaf": params["min_samples_leaf"],

        "Train MAE": -scores["train_MAE"].mean(),
        "Validation MAE": -scores["test_MAE"].mean(),

        "Train MSE": -scores["train_MSE"].mean(),
        "Validation MSE": -scores["test_MSE"].mean(),

        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,

        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),

        "RMSE Gap": val_rmse - train_rmse,

        "Exact Grade Accuracy (%)": (
            np.mean(rounded_predictions == actual) * 100
        ),

        "Within 1 Grade (%)": (
            np.mean(np.abs(actual - predictions) <= 1) * 100
        ),

        "Within 2 Grades (%)": (
            np.mean(np.abs(actual - predictions) <= 2) * 100
        ),

        "Pass/Fail Accuracy (%)": (
            accuracy_score(actual_binary, predicted_binary) * 100
        ),

        "Precision": precision_score(
            actual_binary, predicted_binary, zero_division=0
        ),

        "Recall": recall_score(
            actual_binary, predicted_binary, zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp) if (tn + fp) > 0 else 0
        ),

        "F1 Score": f1_score(
            actual_binary, predicted_binary, zero_division=0
        ),

        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    })

    for fold in range(5):
        rf_fold_results.append({
            "Experiment": name,
            "Fold": fold + 1,
            "Train RMSE": np.sqrt(-scores["train_MSE"][fold]),
            "Validation RMSE": np.sqrt(-scores["test_MSE"][fold]),
            "Train MAE": -scores["train_MAE"][fold],
            "Validation MAE": -scores["test_MAE"][fold],
            "Train R2": scores["train_R2"][fold],
            "Validation R2": scores["test_R2"][fold]
        })

rf_results_df = pd.DataFrame(rf_results)

rf_results_df = rf_results_df.sort_values(
    "Validation RMSE"
).reset_index(drop=True)

rf_fold_df = pd.DataFrame(rf_fold_results)

print("All Random Forest experiments completed.")

Training RF-1...
Training RF-2...
Training RF-3...
Training RF-4...
Training RF-5...
All Random Forest experiments completed.


Regression error comparison

In [9]:
regression_columns = [
    "Experiment",
    "Trees",
    "Max Depth",
    "Min Leaf",
    "Train MAE",
    "Validation MAE",
    "Train MSE",
    "Validation MSE",
    "Train RMSE",
    "Validation RMSE",
    "Train R2",
    "Validation R2",
    "RMSE Gap"
]

print("RANDOM FOREST – REGRESSION ERROR COMPARISON")

display(
    rf_results_df[regression_columns].round(4)
)

RANDOM FOREST – REGRESSION ERROR COMPARISON


,Experiment,Trees,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap
0,RF-4,300,15,2,0.4330,0.9324,0.5355,2.3638,0.7316,1.4733,0.9744,0.8933,0.7417
1,RF-1,100,None,1,0.3492,0.9424,0.3102,2.3961,0.5567,1.4841,0.9852,0.8919,0.9274
2,RF-3,200,10,2,0.4352,0.9351,0.5306,2.4004,0.7284,1.4848,0.9746,0.8915,0.7565
3,RF-2,200,5,1,0.5519,0.9426,0.6653,2.4713,0.8154,1.5064,0.9682,0.8883,0.6910
4,RF-5,300,None,4,0.5974,0.9521,0.9699,2.5059,0.9846,1.5214,0.9535,0.8862,0.5368


Grade prediction accuracy comparison

In [10]:
grade_columns = [
    "Experiment",
    "Exact Grade Accuracy (%)",
    "Within 1 Grade (%)",
    "Within 2 Grades (%)"
]

print("RANDOM FOREST – GRADE ACCURACY COMPARISON")

display(
    rf_results_df[grade_columns].round(2)
)

RANDOM FOREST – GRADE ACCURACY COMPARISON


,Experiment,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%)
0,RF-4,48.10,70.57,89.56
1,RF-1,47.47,68.67,89.56
2,RF-3,46.20,70.89,89.24
3,RF-2,46.52,70.89,88.92
4,RF-5,45.57,70.25,88.92


Pass/fail classification comparison

Confusion matrix comparison table

In [12]:
confusion_columns = [
    "Experiment",
    "TN",
    "FP",
    "FN",
    "TP"
]

print("RANDOM FOREST – CONFUSION MATRIX COMPARISON")

display(
    rf_results_df[confusion_columns]
)

RANDOM FOREST – CONFUSION MATRIX COMPARISON


,Experiment,TN,FP,FN,TP
0,RF-4,102,1,34,179
1,RF-1,102,1,37,176
2,RF-3,102,1,33,180
3,RF-2,102,1,33,180
4,RF-5,102,1,33,180


Overfitting and underfitting assessment

In [13]:
baseline_scores = cross_validate(
    DummyRegressor(strategy="mean"),
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_rmse = np.sqrt(
    -baseline_scores["test_MSE"]
).mean()

def assess_fitting(row):

    gap_ratio = (
        row["RMSE Gap"] / row["Train RMSE"]
        if row["Train RMSE"] > 0 else np.inf
    )

    if row["Validation RMSE"] >= baseline_rmse:
        return "Weak performance / possible underfitting"

    elif gap_ratio > 0.30:
        return "Possible overfitting"

    else:
        return "Reasonable generalization"

rf_results_df["Fitting Assessment"] = (
    rf_results_df.apply(assess_fitting, axis=1)
)

print(f"Baseline Validation RMSE: {baseline_rmse:.4f}")

display(
    rf_results_df[
        [
            "Experiment",
            "Train RMSE",
            "Validation RMSE",
            "RMSE Gap",
            "Fitting Assessment"
        ]
    ].round(4)
)

Baseline Validation RMSE: 4.5635


,Experiment,Train RMSE,Validation RMSE,RMSE Gap,Fitting Assessment
0,RF-4,0.7316,1.4733,0.7417,Possible overfitting
1,RF-1,0.5567,1.4841,0.9274,Possible overfitting
2,RF-3,0.7284,1.4848,0.7565,Possible overfitting
3,RF-2,0.8154,1.5064,0.6910,Possible overfitting
4,RF-5,0.9846,1.5214,0.5368,Possible overfitting


Final Random Forest comparison

In [14]:
final_comparison = rf_results_df.copy()

final_comparison.insert(
    0,
    "Rank",
    np.arange(1, len(final_comparison) + 1)
)

print("=" * 65)
print("FINAL RANDOM FOREST REGRESSION COMPARISON")
print("=" * 65)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 250
):
    display(final_comparison.round(4))

FINAL RANDOM FOREST REGRESSION COMPARISON


,Rank,Experiment,Trees,Max Depth,Min Leaf,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TP,TN,FP,FN,Fitting Assessment
0,1,RF-4,300,15,2,0.4330,0.9324,0.5355,2.3638,0.7316,1.4733,0.9744,0.8933,0.7417,48.1013,70.5696,89.5570,88.9241,0.9944,0.8404,0.9903,0.9109,179,102,1,34,Possible overfitting
1,2,RF-1,100,None,1,0.3492,0.9424,0.3102,2.3961,0.5567,1.4841,0.9852,0.8919,0.9274,47.4684,68.6709,89.5570,87.9747,0.9944,0.8263,0.9903,0.9026,176,102,1,37,Possible overfitting
2,3,RF-3,200,10,2,0.4352,0.9351,0.5306,2.4004,0.7284,1.4848,0.9746,0.8915,0.7565,46.2025,70.8861,89.2405,89.2405,0.9945,0.8451,0.9903,0.9137,180,102,1,33,Possible overfitting
3,4,RF-2,200,5,1,0.5519,0.9426,0.6653,2.4713,0.8154,1.5064,0.9682,0.8883,0.6910,46.5190,70.8861,88.9241,89.2405,0.9945,0.8451,0.9903,0.9137,180,102,1,33,Possible overfitting
4,5,RF-5,300,None,4,0.5974,0.9521,0.9699,2.5059,0.9846,1.5214,0.9535,0.8862,0.5368,45.5696,70.2532,88.9241,89.2405,0.9945,0.8451,0.9903,0.9137,180,102,1,33,Possible overfitting


Select the best Random Forest configuration

In [15]:
best_result = final_comparison.iloc[0]

best_name = best_result["Experiment"]
best_params = rf_experiments[best_name]

print("=" * 60)
print("BEST RANDOM FOREST CONFIGURATION")
print("=" * 60)

print("Experiment:", best_name)
print("Parameters:", best_params)

print("\nREGRESSION METRICS")
print(f"Validation RMSE : {best_result['Validation RMSE']:.4f}")
print(f"Validation MAE  : {best_result['Validation MAE']:.4f}")
print(f"Validation R²   : {best_result['Validation R2']:.4f}")
print(f"RMSE Gap        : {best_result['RMSE Gap']:.4f}")

print("\nGRADE ACCURACY")
print(
    f"Exact Accuracy  : "
    f"{best_result['Exact Grade Accuracy (%)']:.2f}%"
)
print(
    f"Within ±1 Grade : "
    f"{best_result['Within 1 Grade (%)']:.2f}%"
)
print(
    f"Within ±2 Grades: "
    f"{best_result['Within 2 Grades (%)']:.2f}%"
)

print("\nPASS/FAIL CLASSIFICATION")
print(
    f"Accuracy : "
    f"{best_result['Pass/Fail Accuracy (%)']:.2f}%"
)
print(f"Precision: {best_result['Precision']:.4f}")
print(f"Recall   : {best_result['Recall']:.4f}")
print(f"Specificity: {best_result['Specificity']:.4f}")
print(f"F1 Score : {best_result['F1 Score']:.4f}")

print("\nCONFUSION MATRIX")
print("TN:", int(best_result["TN"]))
print("FP:", int(best_result["FP"]))
print("FN:", int(best_result["FN"]))
print("TP:", int(best_result["TP"]))

print("\nFITTING ASSESSMENT")
print(best_result["Fitting Assessment"])

BEST RANDOM FOREST CONFIGURATION
Experiment: RF-4
Parameters: {'n_estimators': 300, 'max_depth': 15, 'min_samples_leaf': 2}

REGRESSION METRICS
Validation RMSE : 1.4733
Validation MAE  : 0.9324
Validation R²   : 0.8933
RMSE Gap        : 0.7417

GRADE ACCURACY
Exact Accuracy  : 48.10%
Within ±1 Grade : 70.57%
Within ±2 Grades: 89.56%

PASS/FAIL CLASSIFICATION
Accuracy : 88.92%
Precision: 0.9944
Recall   : 0.8404
Specificity: 0.9903
F1 Score : 0.9109

CONFUSION MATRIX
TN: 102
FP: 1
FN: 34
TP: 179

FITTING ASSESSMENT
Possible overfitting
